# 进阶实践项目参考答案 01：MRI 肿瘤分割：阈值、边界与不确定性

在 Dice 评价之外，进一步比较阈值、边界误差和概率不确定性。

Kaggle 中先复制到自己的账户，再按任务顺序完成。题目只保留关键填写位置，数据读取、绘图和保存框架已经给出。

## 任务
1. 构造患者级留出
2. 在验证集选择阈值
3. 同时计算 Dice 与边界误差
4. 绘制不确定性图
5. 比较阈值变化带来的取舍

In [1]:
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
from scipy.ndimage import binary_erosion, distance_transform_edt, gaussian_filter
SEED=42; OUT=Path('advanced01_results'); OUT.mkdir(exist_ok=True); rng=np.random.default_rng(SEED)
n,h,w=36,96,96; yy,xx=np.mgrid[:h,:w]; target=[]; prob=[]; patient=np.repeat(np.arange(12),3)
for i in range(n):
    cy,cx=rng.integers(28,68,2); ry,rx=rng.integers(10,23,2); mask=(((yy-cy)/ry)**2+((xx-cx)/rx)**2<1)
    p=gaussian_filter(mask.astype(float),2.2)+rng.normal(0,.08,(h,w)); target.append(mask); prob.append(np.clip(p,0,1))
target=np.asarray(target); prob=np.asarray(prob); train=patient<8; val=(patient>=8)&(patient<10); test=patient>=10
def dice(a,b): return float((2*(a&b).sum()+1)/((a.sum()+b.sum())+1))
thresholds=np.arange(.25,.76,.05); val_scores=[np.mean([dice(prob[i]>=t,target[i]) for i in np.where(val)[0]]) for t in thresholds]; best=float(thresholds[int(np.argmax(val_scores))])
pred=prob[test]>=best; truth=target[test]; dices=[dice(a,b) for a,b in zip(pred,truth)]
def boundary_distance(a,b):
    ba=a^binary_erosion(a); bb=b^binary_erosion(b); return float((distance_transform_edt(~bb)[ba].mean()+distance_transform_edt(~ba)[bb].mean())/2)
dist=[boundary_distance(a,b) for a,b in zip(pred,truth)]; uncertainty=1-np.abs(prob[test][0]-.5)*2
fig,ax=plt.subplots(1,4,figsize=(12,3)); ax[0].imshow(prob[test][0],cmap='gray'); ax[0].set_title('probability'); ax[1].imshow(truth[0],cmap='gray'); ax[1].set_title('target'); ax[2].imshow(pred[0],cmap='gray'); ax[2].set_title(f'prediction t={best:.2f}'); ax[3].imshow(uncertainty,cmap='magma'); ax[3].set_title('uncertainty'); [a.axis('off') for a in ax]; fig.tight_layout(); fig.savefig(OUT/'advanced01_summary.png',dpi=150); plt.close(fig)
result={'best_threshold':best,'test_dice_mean':float(np.mean(dices)),'boundary_distance_mean':float(np.mean(dist)),'patients':{'train':8,'validation':2,'test':2}}; (OUT/'advanced01_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8'); print(result)


{'best_threshold': 0.44999999999999996, 'test_dice_mean': 0.9769748883989026, 'boundary_distance_mean': 0.3456778266167051, 'patients': {'train': 8, 'validation': 2, 'test': 2}}
